# Première analyse descriptive du fichier CSV
Ce notebook donne une vue d'ensemble du fichier : dimensions, colonnes, valeurs manquantes, statistiques, doublons et graphiques simples.

**À faire :** placez `consommation-annuelle-par-iris.csv` dans le même dossier que ce notebook, puis exécutez la cellule ci-dessous. Le code suppose un CSV séparé par une virgule ou un point-virgule.

In [ ]:
# Analyse descriptive simple de consommation-annuelle-par-iris.csv
# Placez le fichier CSV dans le même dossier que ce notebook, puis exécutez cette cellule.

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# 1. Charger le fichier (détection simple du séparateur , ou ;)
fichier = Path("consommation-annuelle-par-iris.csv")
if not fichier.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {fichier.resolve()}\n"
        "Placez le CSV dans le même dossier que ce notebook."
    )

with open(fichier, "r", encoding="utf-8-sig") as f:
    extrait = f.read(10000)
separateur = ";" if extrait.count(";") > extrait.count(",") else ","

df = pd.read_csv(fichier, sep=separateur, encoding="utf-8-sig", low_memory=False)

# 2. Vue d'ensemble : taille, colonnes, types et premières lignes
print("=== APERÇU GÉNÉRAL ===")
print(f"Nombre de lignes : {df.shape[0]:,}")
print(f"Nombre de colonnes : {df.shape[1]}")
print(f"Séparateur utilisé : {repr(separateur)}")
print("\n=== COLONNES ET TYPES ===")
display(pd.DataFrame({
    "type": df.dtypes.astype(str),
    "valeurs_distinctes": df.nunique(dropna=True),
    "valeurs_manquantes": df.isna().sum(),
    "% manquant": (df.isna().mean() * 100).round(2)
}).sort_values("% manquant", ascending=False))
print("\n=== PREMIÈRES LIGNES ===")
display(df.head(10))

# 3. Résumé statistique : variables numériques et catégorielles
print("\n=== STATISTIQUES DES VARIABLES NUMÉRIQUES ===")
display(df.describe().T)
print("\n=== VARIABLES TEXTUELLES : EXEMPLES DE MODALITÉS ===")
for col in df.select_dtypes(include=["object", "category", "string"]).columns:
    print(f"\n{col} — {df[col].nunique(dropna=True)} modalités distinctes")
    print(df[col].value_counts(dropna=False).head(5).to_string())

# 4. Contrôles simples de qualité
print("\n=== QUALITÉ DES DONNÉES ===")
print(f"Lignes entièrement dupliquées : {df.duplicated().sum():,}")
colonnes_num = df.select_dtypes(include="number").columns
if len(colonnes_num):
    print("\nNombre de valeurs négatives par variable numérique :")
    display((df[colonnes_num] < 0).sum().to_frame("valeurs_négatives"))

# 5. Graphiques simples pour repérer les distributions
# On limite le nombre de graphiques aux 6 premières variables numériques.
if len(colonnes_num):
    for col in colonnes_num[:6]:
        valeurs = df[col].dropna()
        if len(valeurs) == 0:
            continue
        plt.figure(figsize=(7, 3.5))
        plt.hist(valeurs, bins=30, edgecolor="black")
        plt.title(f"Distribution de « {col} »")
        plt.xlabel(col)
        plt.ylabel("Nombre de lignes")
        plt.tight_layout()
        plt.show()

# 6. Repérer les colonnes qui pourraient représenter une année ou un code IRIS
print("\n=== COLONNES POSSIBLEMENT TEMPORELLES OU GÉOGRAPHIQUES ===")
candidates = [
    col for col in df.columns
    if any(mot in col.lower() for mot in ["annee", "année", "year", "iris", "commune", "code_insee"])
]
if candidates:
    for col in candidates:
        print(f"\n{col} — exemples : {df[col].dropna().astype(str).unique()[:8].tolist()}")
else:
    print("Aucune colonne reconnue automatiquement : vérifiez les noms de colonnes ci-dessus.")

print("\nAnalyse terminée. Les résultats sont descriptifs : vérifiez les unités et le sens des colonnes avant toute conclusion.")


FileNotFoundError: Fichier introuvable : C:\Users\dufau\Documents\GitHub\Projet_Impact_IA_Generative\Data\Data\consommation-annuelle-par-iris.csv
Placez le CSV dans le même dossier que ce notebook.